In [1]:
import os
import sys
from pyspark.sql import SparkSession
from pyspark.sql.window import Window
from pyspark.sql import functions as F
from pyspark.sql.functions import dense_rank, desc,when
from pyspark.sql.functions import dense_rank, desc
from pyspark.sql.types import StructType, StructField, StringType, IntegerType,CharType,ArrayType
from itertools import chain

In [2]:
# 1. Point directly to your installed JDK 17 path
os.environ["JAVA_HOME"] = r"C:\Program Files\Eclipse Adoptium\jdk-17.0.20.8-hotspot"
os.environ["PATH"] = os.path.join(os.environ["JAVA_HOME"], "bin") + os.pathsep + os.environ["PATH"]

# Ensure PySpark uses the active Python interpreter
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

# 2. Now import and start Spark


spark = SparkSession.builder \
    .appName("pyspark_ylep") \
    .master("local[*]") \
    .config("spark.driver.memory", "8g") \
    .config("spark.driver.maxResultSize", "4g") \
    .config("spark.sql.shuffle.partitions", "8") \
    .config("spark.driver.bindAddress", "127.0.0.1") \
    .config("spark.sql.autoBroadcastJoinThreshold", "64MB") \
    .config("spark.local.dir", "D:/spark_tmp") \
    .config("spark.sql.execution.pyspark.udf.faulthandler.enabled", "true") \
    .config("spark.python.worker.faulthandler.enabled", "true") \
    .config("spark.sql.execution.pythonUDF.arrow.enabled", "false") \
    .config("spark.sql.autoBroadcastJoinThreshold", "-1")\
    .getOrCreate()

c:\Users\User\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


In [3]:
df_read_business_data = spark.read.json("D:\\A_Data_engineer\\Project_Yelp\\Yelp JSON\\yelp_academic_dataset_business.json")

In [4]:
df_categories =df_read_business_data.select(F.col("categories"))

In [5]:
df_read_dim_category=spark.read.parquet("D:/A_Data_engineer/Project_Yelp/store_dimesnion_user/dim_category")

In [ ]:
df_with_map = df_read_dim_category.withColumn("Category_pk_sk", F.create_map(F.col("Category_name"), F.col("Category_pk_sk")))


In [91]:
def get_category_name(text):
    if text is None:
        return "Not Specified", ["Not Specified"]
    list_of_element = text.split(",")
    category_name = list_of_element[0].strip()
    del list_of_element[0]
    if len(list_of_element) == 0:
        return category_name, ["Not Specified"]
    list_of_element_without_space = [st.lower().strip() for st in list_of_element]
    return category_name, list_of_element_without_space

schema = StructType([
    StructField("Categorie_name", StringType(), True),
    StructField("tag_name", ArrayType(StringType()), True),
])
get_category_name_udf = F.udf(get_category_name, schema)

# Step 1: apply UDF, extract fields, then CACHE + materialize — no explode yet
df_after_udf = (
    df_categories
    .withColumn("new_eilte", get_category_name_udf(F.col("categories")))
    .withColumn("tag_name_arr", F.col("new_eilte.tag_name"))
    .drop("new_eilte", "categories")
    .cache()
)

#df_after_udf_2 = df_after_udf.withColumn("Subcategory_pk_sk", F.monotonically_increasing_id() + 1) 
df_after_udf.count()  # force materialization now

# Step 2: explode AFTER the UDF stage is fully materialized
df_elite_2 = (
    df_after_udf
    .withColumn("tag_name", F.explode(F.col("tag_name_arr")))
    .drop("tag_name_arr")
)

In [93]:
df_elite = df_elite_2.distinct()

In [94]:
df_elite.orderBy(F.col("tag_name")).show(n=200)

+--------------------+
|            tag_name|
+--------------------+
|          & probates|
|         3d printing|
|       Not Specified|
|          acai bowls|
|         accessories|
|         accountants|
|      acne treatment|
|         active life|
|         acupuncture|
|  addiction medicine|
|   adoption services|
|               adult|
|     adult education|
| adult entertainment|
|         advertising|
|      aerial fitness|
|        aerial tours|
|       aestheticians|
|              afghan|
|             african|
|   air duct cleaning|
|    aircraft dealers|
|    aircraft repairs|
|            airlines|
|     airport lounges|
|    airport shuttles|
|   airport terminals|
|            airports|
|             airsoft|
|          allergists|
|alternative medicine|
|amateur sports teams|
|      american (new)|
|american (traditi...|
|     amusement parks|
|   anesthesiologists|
|animal physical t...|
|     animal shelters|
|            antiques|
|    apartment agents|
|          

In [100]:
df_final_2 = df_elite.withColumn("Subcategory_pk_sk", F.monotonically_increasing_id() + 1) 

In [101]:
df_final_2.orderBy(F.col("tag_name")).show(n=1000)

+--------------------+-----------------+
|            tag_name|Subcategory_pk_sk|
+--------------------+-----------------+
|          & probates|              479|
|         3d printing|             1236|
|       Not Specified|              419|
|          acai bowls|              542|
|         accessories|              126|
|         accountants|               73|
|      acne treatment|              444|
|         active life|                6|
|         acupuncture|              910|
|  addiction medicine|              356|
|   adoption services|             1100|
|               adult|              131|
|     adult education|              707|
| adult entertainment|              832|
|         advertising|              307|
|      aerial fitness|             1010|
|        aerial tours|              588|
|       aestheticians|              196|
|              afghan|              347|
|             african|              701|
|   air duct cleaning|              582|
|    aircraft de

In [73]:
df_join_elite_cate = df_elite.join(df_with_map, df_elite.Categorie_name == df_with_map.Category_name, how="inner")

In [75]:
df_elite_4 =  df_join_elite_cate.withColumn("Category_pk_sk" , F.element_at(F.col("Category_pk_sk"),F.col("Categorie_name")) )

In [74]:
df_join_elite_cate

DataFrame[Categorie_name: string, tag_name: string, Category_name: string, Category_pk_sk: map<string,bigint>]

In [76]:
df_elite_4.show()

+-----------------+------------------+-----------------+--------------+
|   Categorie_name|          tag_name|    Category_name|Category_pk_sk|
+-----------------+------------------+-----------------+--------------+
|      3D Printing|    local services|      3D Printing|           273|
|      3D Printing|       hobby shops|      3D Printing|           273|
|      3D Printing|          shopping|      3D Printing|           273|
|ATV Rentals/Tours|   hotels & travel|ATV Rentals/Tours|           679|
|ATV Rentals/Tours|       active life|ATV Rentals/Tours|           679|
|ATV Rentals/Tours|           boating|ATV Rentals/Tours|           679|
|ATV Rentals/Tours|             tours|ATV Rentals/Tours|           679|
|ATV Rentals/Tours|           fishing|ATV Rentals/Tours|           679|
|ATV Rentals/Tours|        automotive|ATV Rentals/Tours|           679|
|ATV Rentals/Tours| motorcycle repair|ATV Rentals/Tours|           679|
|       Acai Bowls|      local flavor|       Acai Bowls|        

In [ ]:
df_final_2.orderBy(F.col("tag_name")).show(n=1000)

+--------------+--------------------+-------------------+-----------------+
|Category_pk_sk|      Categorie_name|           tag_name|Subcategory_pk_sk|
+--------------+--------------------+-------------------+-----------------+
|           405|      Bankruptcy Law|         & probates|             4104|
|          1087|        Business Law|         & probates|             7102|
|           918|Divorce & Family Law|         & probates|            12458|
|           460|      Employment Law|         & probates|            13798|
|           122| Estate Planning Law|         & probates|            13928|
|           507|  Financial Advising|         & probates|            15509|
|           836|  General Litigation|         & probates|            17861|
|          1024|       Home Services|         & probates|            21343|
|            37|             Lawyers|         & probates|            24471|
|            60|      Legal Services|         & probates|            24561|
|           

In [48]:
df_final.orderBy(F.col("Subcategory_pk_sk")).show()

+-----------------+--------------+----------------+
|Subcategory_pk_sk|Category_pk_sk|subcategory_name|
+-----------------+--------------+----------------+
+-----------------+--------------+----------------+



In [27]:
df_elite_4.show()

+--------------+-----------------+--------------------+-------------+--------------+
|Categorie_name|Subcategory_pk_sk|            tag_name|Category_name|Category_pk_sk|
+--------------+-----------------+--------------------+-------------+--------------+
|   3D Printing|      94489283627|      local services|  3D Printing|           273|
|   3D Printing|      94489283627|         hobby shops|  3D Printing|           273|
|   3D Printing|      94489283627|            shopping|  3D Printing|           273|
|   Active Life|               51|fitness & instruc...|  Active Life|           143|
|   Active Life|               51|            trainers|  Active Life|           143|
|   Active Life|               51|                gyms|  Active Life|           143|
|   Active Life|               51|                yoga|  Active Life|           143|
|   Active Life|              169|   basketball courts|  Active Life|           143|
|   Active Life|              169|   stadiums & arenas|  Active L

In [ ]:
def get_category_name(text):

    if text is None:
       return "Not Specified", ["Not Specified"]
    
    list_of_element = text.split(",")
    category_name=list_of_element[0].strip()
    del list_of_element[0]
    if len(list_of_element) == 0:
        return category_name, ["Not Specified"]

    list_of_element_without_space=[st.lower().strip() for st in list_of_element]

    return category_name,list_of_element_without_space  

schema = StructType([
    StructField("Categorie_name", StringType(), True),
    StructField("tag_name", ArrayType(StringType()), True),  
])

get_category_name_udf = F.udf(get_category_name, schema)

df_elite = (
    df_categories.withColumn("new_eilte", get_category_name_udf(F.col("categories")))
     .withColumn("Categorie_name", F.col("new_eilte.Categorie_name"))
     .withColumn("tag_name", F.explode(F.col("new_eilte.tag_name")))
).drop(F.col("new_eilte"),F.col("categories"))


df_elite_3.show()

In [25]:
def make_array(x):
    return [str(x), str(x+1), str(x+2)] if x is not None else []

array_udf = F.udf(make_array, ArrayType(StringType()))

test_df = spark.range(150000)

# Step 1: UDF only, then CACHE — this forces materialization, breaking the pipelined stage
df_with_arr = test_df.withColumn("arr", array_udf(F.col("id"))).cache()
df_with_arr.count()  # force the cache to materialize now, isolated from explode

# Step 2: explode AFTER caching — now explode operates on already-materialized data, not a live UDF pipeline
result = df_with_arr.withColumn("exploded", F.explode(F.col("arr"))).count()
print(f"SUCCESS: {result}")

SUCCESS: 450000


In [29]:
df_elite.columns

['Categorie_name', 'tag_name']

In [12]:
df_join_elite_cate.show()


+--------------+--------------------+-------------+--------------------+
|Categorie_name|            tag_name|Category_name|      Category_pk_sk|
+--------------+--------------------+-------------+--------------------+
|   3D Printing|      local services|  3D Printing|{3D Printing -> 273}|
|   3D Printing|         hobby shops|  3D Printing|{3D Printing -> 273}|
|   3D Printing|            shopping|  3D Printing|{3D Printing -> 273}|
|   Active Life|fitness & instruc...|  Active Life|{Active Life -> 143}|
|   Active Life|            trainers|  Active Life|{Active Life -> 143}|
|   Active Life|                gyms|  Active Life|{Active Life -> 143}|
|   Active Life|                yoga|  Active Life|{Active Life -> 143}|
|   Active Life|   basketball courts|  Active Life|{Active Life -> 143}|
|   Active Life|   stadiums & arenas|  Active Life|{Active Life -> 143}|
|   Active Life|arts & entertainment|  Active Life|{Active Life -> 143}|
|   Active Life|professional spor...|  Active Life|

In [ ]:
dict_longitude =dict()

for i,j in df_read_dim_category.collect():
   dict_longitude[i] = j





In [ ]:
# Declare an empty Dictionary
dict = {}

# Convert PySpark DataFrame to Pandas 
# DataFrame
df = df_read_dim_category.toPandas()

# Traverse through each column
for column in df.columns:

    # Add key as column_name and
    # value as list of column values
    dict["Category_name"] = df["Category_pk_sk"].values.tolist()

# Print the dictionary
print(dict)

In [ ]:
map_expr = F.create_map( [F.lit(x) for x in chain(*df_with_map.items())] )

df_elite_4 =  df_elite_3.withColumn("Category_pk_sk" , map_expr.getItem(F.col("Categorie_name")))

PySparkAttributeError: [ATTRIBUTE_NOT_SUPPORTED] Attribute `F` is not supported.

In [ ]:
collected_data = df_with_map.collect()
dictionary = {row["Category_name"]: row["Category_pk_sk"] for row in collected_data}

print("Dictionary:", dictionary)
print("Type:", type(dictionary))

TypeError: cannot use 'pyspark.sql.classic.column.Column' as a dict key (unhashable type: 'Column')

In [ ]:
df_read_dim_category.columns

['Category_name', 'Category_pk_sk']

In [17]:
df_read_dim_category.show()

+--------------------+--------------+
|       Category_name|Category_pk_sk|
+--------------------+--------------+
|             Doctors|             1|
|    Shipping Centers|             2|
|   Department Stores|             3|
|         Restaurants|             4|
|            Brewpubs|             5|
|             Burgers|             6|
|      Sporting Goods|             7|
|          Synagogues|             8|
|                Pubs|             9|
|Ice Cream & Froze...|            10|
|   Department Stores|            11|
|          Vietnamese|            12|
|American (Traditi...|            13|
|   General Dentistry|            14|
|                Food|            15|
|          Sushi Bars|            16|
|          Automotive|            17|
|          Vape Shops|            18|
|          Automotive|            19|
|              Korean|            20|
+--------------------+--------------+
only showing top 20 rows
